# Thu th?p ph? l?c t? VBPL

Notebook ch? t?i v? l?u file **tr?n m?y**. Kh?ng n?p Neo4j/Qdrant, kh?ng g?i embedding/LLM, kh?ng upload Cloudflare R2. C?c d?ch v? cloud hi?n c? gi? c?u h?nh ri?ng.

M? b?ng kernel ?? c?i `requirements-appendix.txt`. M?i cell truy c?p m?ng ??u t?t m?c ??nh. Ch?y l?i s? ki?m tra hash v? ti?p t?c manifest ?? c?.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "labor_rag").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")  # Kh?ng hi?n th? gi? tr? kh?a
import pandas as pd
from IPython.display import display
from labor_rag.appendices.collect import Collector, load_targets
from labor_rag.appendices.catalog import RAW_DIR, INVENTORY_DIR, load_manifest, write_json, valid_artifact

ENABLE_NETWORK = False
RUN_SAMPLE = False
RUN_BATCH = False
REFRESH_EXISTING = False
SAMPLE_DOCUMENT = "09-2020-tt-bldtbxh"
print("Th? m?c file:", RAW_DIR)
print("Truy c?p m?ng:", ENABLE_NETWORK)

## 1. Ph?m vi v? ngu?n

38 m?c ki?m k? ban ??u; ngu?n s?a ??i ???c b? sung ri?ng khi link c? tr? sang v?n b?n kh?c. `source_overrides.csv` s?a ItemID theo ??ng s? hi?u; `supplemental_sources.csv` gi? ngu?n s?a ??i v?i danh t?nh ri?ng. S? m?c thu th?p kh?ng ph?i s? ph? l?c. Ch?a d?ng ng?y t?i l?m ng?y hi?u l?c.

In [ ]:
targets = load_targets()
base = [t for t in targets if not t.get("supplemental")]
print(f"{len(base)} m?c g?c; {len(targets)-len(base)} ngu?n s?a ??i b? sung")
display(pd.DataFrame(targets).reindex(columns=["document", "document_id", "item_id", "source_url", "amends", "notes"]))

In [ ]:
groups = pd.read_csv(INVENTORY_DIR / "file_groups.csv").fillna("")
refs = pd.read_csv(INVENTORY_DIR / "appendix_references.csv").fillna("")
print("Nh?m file / d?n chi?u (kh?ng ??ng ngh?a v?i ph? l?c th?c t?):", len(groups), len(refs))
display(groups.head(10))
display(refs.head(10))

## 2. Ch?y th? TT 09/2020

B?t `ENABLE_NETWORK` v? `RUN_SAMPLE` ? cell c?u h?nh ?? t?i. HTTP t?i ?a hai y?u c?u ??ng th?i, kho?ng ngh? d?ng chung v? retry h?u h?n. T?n file l?y t? metadata giao di?n m?i; kh?ng ?o?n t? AppFile GUID.

In [ ]:
if ENABLE_NETWORK and RUN_SAMPLE:
    sample = next(t for t in targets if t["document_id"] == SAMPLE_DOCUMENT)
    Collector().collect([sample], refresh=REFRESH_EXISTING)
else:
    print("B? qua t?i m?u: ch?a b?t c? ENABLE_NETWORK v? RUN_SAMPLE.")
manifest = load_manifest()
sample_result = manifest["documents"].get(SAMPLE_DOCUMENT, {})
display(pd.DataFrame(sample_result.get("files", [])).reindex(columns=["artifact_id", "file_name", "format", "bytes", "source_url", "path"]))

## 3. Thu th?p theo l? / ti?p t?c

B?t `RUN_BATCH` ?? ch?y to?n b? ph?m vi. Manifest ???c ghi sau m?i v?n b?n ho?n th?nh; l?n sau b? qua file h?p l?. C? th? ??t `BATCH_DOCUMENTS` ?? gi?i h?n v?n b?n. L?i 404, file sai ??nh d?ng ho?c sai s? hi?u lu?n gi? tr?ng th?i l?i.

In [ ]:
BATCH_DOCUMENTS = []  # R?ng = m?i ngu?n; ho?c danh s?ch document_id
batch_targets = [t for t in targets if not BATCH_DOCUMENTS or t["document_id"] in BATCH_DOCUMENTS]
if ENABLE_NETWORK and RUN_BATCH:
    Collector().collect(batch_targets, refresh=REFRESH_EXISTING)
else:
    print("B? qua t?i l?: ch?a b?t c? ENABLE_NETWORK v? RUN_BATCH.")

## 4. K?t qu?, file tr?ng v? ngu?n c?n x? l?

Kh? tr?ng v?t l? theo SHA-256, v?n gi? t?t c? quan h? v?n b?n?file. Thu th?p th?nh c?ng kh?ng ??ng ngh?a ?? tr?ch xu?t ph? l?c th?nh c?ng. Ph?n nh?n di?n ph? l?c n?m ? pipeline tr?ch xu?t b?n ngo?i notebook n?y.

In [ ]:
manifest = load_manifest()
rows = [{"document": d["document"], "download_status": d.get("download_status"), "files": len(d.get("files", [])), "cache_hit": d.get("cache_hit", False), "errors": " | ".join(d.get("errors", []))} for d in manifest["documents"].values()]
display(pd.DataFrame(rows))
files = [dict(f, document=d["document"]) for d in manifest["documents"].values() for f in d.get("files", [])]
if files:
    df = pd.DataFrame(files)
    display(df[df.duplicated("artifact_id", keep=False)].reindex(columns=["document", "artifact_id", "file_name"]))
    invalid = [f for f in files if not valid_artifact(f)]
    print("File c?n t?i l?i do thi?u/sai hash:", len(invalid))
    display(pd.DataFrame(invalid))

In [ ]:
from labor_rag.appendices.catalog import PROCESSED_DIR, read_json
extraction = read_json(PROCESSED_DIR / "manifest.json", {})
if extraction:
    print("K?t qu? pipeline tr?ch xu?t hi?n c?:", extraction.get("appendices"), "ph?n nh?n di?n;", extraction.get("chunks"), "chunk")
    display(pd.DataFrame(extraction.get("documents", [])))
else:
    print("Ch?a ch?y pipeline tr?ch xu?t; xem docs/APPENDICES.md.")

## 5. Xu?t manifest b?n giao

Manifest ??y ?? v?n ? `data/raw/appendix/manifest.json`. B?n xu?t sau ??y ch?a tr?ng th?i t?i, ngu?n, hash v? danh s?ch l?i ?? pipeline tr?ch xu?t s? d?ng. C?u h?nh R2 kh?ng t?c ??ng c?c cell n?y.

In [ ]:
export_path = RAW_DIR / "collection_manifest.json"
write_json(export_path, load_manifest())
print("?? xu?t:", export_path)